# 🔬 Notebook 3: Discord — Deep Dive: Fan-out, Presence, Voice

## 🛠️ Setup

```bash
cd 06-system-designs/discord
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive — fanout for a 100k-member channel

Naively: server receives one message, pushes it to 100,000 WebSocket connections.
But those connections are spread across 300 gateway instances. We need **pub/sub by channel**.

```
  client ──▶ gateway A ──▶ bus (topic="channel-123") ──▶ gateway A, B, C, ..., Z
                                                        └────┬────┘
                                                             ▼
                                               each gateway pushes to
                                               its locally-connected
                                               members of that channel
```

Each gateway maintains, for each channel, the set of its connected members.
A message produces 1 bus event + N socket writes where N = members-on-this-shard.

We'll simulate this below.

In [ ]:
from collections import defaultdict

# A simplified sync model — the point is fan-out, not asyncio itself.
class FakeSocket:
    def __init__(self, user_id): self.user_id = user_id; self.inbox = []
    def send(self, msg): self.inbox.append(msg)

class Gateway:
    def __init__(self, name):
        self.name = name
        # channel_id -> set[FakeSocket]
        self.subs: "dict[int, set[FakeSocket]]" = defaultdict(set)
    def subscribe(self, channel_id, sock): self.subs[channel_id].add(sock)
    def deliver(self, channel_id, msg):
        # fan out to local subscribers only — O(local) not O(global)
        for s in self.subs[channel_id]:
            s.send(msg)

class Bus:
    def __init__(self): self.gateways: "list[Gateway]" = []
    def publish(self, channel_id, msg):
        # In production the bus is Kafka / Redis pub-sub.
        # Here we synchronously notify every gateway instance.
        for g in self.gateways:
            g.deliver(channel_id, msg)

bus = Bus()
gws = [Gateway(f"gw{i}") for i in range(3)]
bus.gateways = gws

# 10 users spread across 3 gateways, all subscribed to channel 123
socks = [FakeSocket(i) for i in range(10)]
for i, s in enumerate(socks):
    gws[i % 3].subscribe(123, s)

bus.publish(123, {"content": "hello channel 123"})
for s in socks:
    print(f"user {s.user_id}: {s.inbox}")


## Deep dive — presence at scale

Naive presence (every user online-status broadcast to all friends) is O(friends) writes per
state change. Discord famously solved it with **per-guild presence sessions** plus a gossip
layer — lazy presence.

Key trick: you only need presence for **people you actually see**. So:
- On join/open channel: subscribe to presence events for members you can see.
- On leave: unsubscribe.

This bounds presence fan-out per user to ~100 friends instead of entire guild.

## Deep dive — voice (why WebRTC/UDP)

- TCP retransmits lost packets → audio would stutter.
- For voice, **it's better to drop a packet than retransmit**.
- UDP + WebRTC + Opus codec gives ~40ms end-to-end latency.
- A **Selective Forwarding Unit (SFU)** receives streams from each speaker and forwards only
  to listeners in the voice channel (doesn't transcode — low CPU, low latency).
